In [1]:
# =============================================================================
# IMPORTS Y CONFIGURACIÓN
# =============================================================================
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import json
import gc
import psutil
import os
from pathlib import Path
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.decomposition import PCA
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuración para plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 7)

# Paths
RAW_DIR = Path("../data/raw/cluvi/")
OUTPUT_DIR = "../data/process/"  # Adaptado si es necesario
REPORT_DIR = Path("../reports/eda_cluvi/")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("EDA COMPLETO Y MEJORADO - DATASET CLUVI (MENÚS Y CATEGORÍAS)")
print("="*80)

EDA COMPLETO Y MEJORADO - DATASET CLUVI (MENÚS Y CATEGORÍAS)


In [2]:
# Paths
OUTPUT_DIR = "../data/process/"
sentences_path = f"{OUTPUT_DIR}sentences_final.parquet"
entities_path = f"{OUTPUT_DIR}entities.parquet"
main_path = f"{OUTPUT_DIR}main_cleaned.parquet"
REPORT_DIR = Path("../reports/eda/")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Análisis de cada uno de los DataFrames usando Polars
print("\n======= Análisis rápido de los DataFrames con polars =======\n")

# Sentences DataFrame
try:
    sentences_df = pl.read_parquet(sentences_path)
    print(f"Sentences DF: {sentences_df.shape[0]:,} filas, {sentences_df.shape[1]} columnas")
    print(sentences_df.head(3))
    print(sentences_df.describe())
except Exception as e:
    print(f"No se pudo leer {sentences_path}: {e}")

print("\n-------------------------------------\n")



======= Análisis rápido de los DataFrames con polars =======

Sentences DF: 3,399,384 filas, 3 columnas
shape: (3, 3)
┌─────────────┬─────────────────────────────────┬──────────────────────────────┐
│ sentence_id ┆ sentence                        ┆ tokens                       │
│ ---         ┆ ---                             ┆ ---                          │
│ str         ┆ str                             ┆ list[str]                    │
╞═════════════╪═════════════════════════════════╪══════════════════════════════╡
│ SENT_00001  ┆ peel and chop the tomato and o… ┆ ["peel", "and", … "."]       │
│ SENT_00002  ┆ cook and stir butter, mushroom… ┆ ["cook", "and", … "."]       │
│ SENT_00003  ┆ mix shortening, sugar and eggs… ┆ ["mix", "shortening", … "."] │
└─────────────┴─────────────────────────────────┴──────────────────────────────┘
shape: (9, 4)
┌────────────┬─────────────┬─────────────────────────────────┬────────────┐
│ statistic  ┆ sentence_id ┆ sentence                        ┆

In [9]:
# Mostrar toda la información (no truncada) del registro con id SENT_22548,
# especialmente para columnas tipo lista o texto largo.
sent_id = "SENT_22548"
if 'sentences_df' in locals():
    info_sent_22548 = sentences_df.filter(pl.col("sentence_id") == sent_id)
    if info_sent_22548.height == 0:
        print(f"No se encontró información para el id {sent_id}.")
    else:
        print(f"Información para el id {sent_id}:")
        # Imprimir cada columna por separado para evitar truncamientos
        for row in info_sent_22548.to_dicts():
            for key, value in row.items():
                print(f"{key}: {value}\n{'-'*50}")
else:
    print("El DataFrame 'sentences_df' no está cargado.")


Información para el id SENT_22548:
sentence_id: SENT_22548
--------------------------------------------------
sentence: 1 c. sour cream, 2 eggs, slightly beaten, 1/2 c. water, 1 tsp. sugar, 1 c. butter, melted, 1 c. sugar, 2 pkg. dry active yeast, 4 c. flour Scald sour cream and put into bowl. Add butter eggs and 1 cup of sugar. Mix slightly.
--------------------------------------------------
tokens: ['1', 'c', 'sour', 'cream', '2', 'eggs', 'slightly', 'beaten', '1', '2', 'c', 'water', '1', 'tsp', 'sugar', '1', 'c', 'butter', 'melted', '1', 'c', 'sugar', '2', 'pkg', 'dry', 'active', 'yeast', '4', 'c', 'flour', 'scald', 'sour', 'cream', 'and', 'put', 'into', 'bowl', 'add', 'butter', 'eggs', 'and', '1', 'cup', 'of', 'sugar', 'mix', 'slightly']
--------------------------------------------------


In [3]:

# Entities DataFrame
try:
    entities_df = pl.read_parquet(entities_path)
    print(f"Entities DF: {entities_df.shape[0]:,} filas, {entities_df.shape[1]} columnas")
    print(entities_df.head(3))
    print(entities_df.describe())
except Exception as e:
    print(f"No se pudo leer {entities_path}: {e}")

print("\n-------------------------------------\n")


Entities DF: 695,252 filas, 4 columnas
shape: (3, 4)
┌────────────┬───────────────┬─────────────┬───────────────┐
│ entity_id  ┆ entity        ┆ type_entity ┆ iob_tag       │
│ ---        ┆ ---           ┆ ---         ┆ ---           │
│ str        ┆ str           ┆ str         ┆ str           │
╞════════════╪═══════════════╪═════════════╪═══════════════╡
│ UENT_00001 ┆ flour         ┆ FOOD        ┆ B-FOOD        │
│ UENT_00002 ┆ milk          ┆ FOOD        ┆ B-FOOD        │
│ UENT_00003 ┆ chicken broth ┆ FOOD        ┆ B-FOOD I-FOOD │
└────────────┴───────────────┴─────────────┴───────────────┘
shape: (9, 5)
┌────────────┬────────────┬────────┬─────────────┬─────────────────────────────┐
│ statistic  ┆ entity_id  ┆ entity ┆ type_entity ┆ iob_tag                     │
│ ---        ┆ ---        ┆ ---    ┆ ---         ┆ ---                         │
│ str        ┆ str        ┆ str    ┆ str         ┆ str                         │
╞════════════╪════════════╪════════╪═════════════╪══════════

In [4]:

# Main DataFrame
try:
    main_df = pl.read_parquet(main_path)
    print(f"Main DF: {main_df.shape[0]:,} filas, {main_df.shape[1]} columnas")
    print(main_df.head(3))
    print(main_df.describe())
except Exception as e:
    print(f"No se pudo leer {main_path}: {e}")



Main DF: 27,296,077 filas, 7 columnas
shape: (3, 7)
┌─────────────────┬─────────────┬────────────┬────────────┬──────────┬─────────────┬───────────┐
│ entity_local_id ┆ sentence_id ┆ entity_id  ┆ char_start ┆ char_end ┆ token_start ┆ token_end │
│ ---             ┆ ---         ┆ ---        ┆ ---        ┆ ---      ┆ ---         ┆ ---       │
│ str             ┆ str         ┆ str        ┆ i64        ┆ i64      ┆ i64         ┆ i64       │
╞═════════════════╪═════════════╪════════════╪════════════╪══════════╪═════════════╪═══════════╡
│ ENT_00001       ┆ SENT_00001  ┆ UENT_00035 ┆ 29         ┆ 34       ┆ 6           ┆ 6         │
│ ENT_00002       ┆ SENT_00001  ┆ UENT_00303 ┆ 68         ┆ 75       ┆ 14          ┆ 14        │
│ ENT_00003       ┆ SENT_00001  ┆ UENT_00042 ┆ 77         ┆ 83       ┆ 16          ┆ 16        │
└─────────────────┴─────────────┴────────────┴────────────┴──────────┴─────────────┴───────────┘
shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬─────────

In [ ]:
# Mostrar toda la información (no truncada) del registro con id SENT_22548,
# especialmente para columnas tipo lista o texto largo.
sent_id = "SENT_22548"
if 'main_df' in locals():
    info_sent_22548 = main_df.filter(pl.col("sentence_id") == sent_id)
    if info_sent_22548.height == 0:
        print(f"No se encontró información para el id {sent_id}.")
    else:
        print(f"Información para el id {sent_id}:")
        # Imprimir cada columna por separado para evitar truncamientos
        for row in info_sent_22548.to_dicts():
            for key, value in row.items():
                print(f"{key}: {value}\n{'-'*50}")
else:
    print("El DataFrame 'main_df' no está cargado.")


Información para el id SENT_22548:
entity_local_id: ENT_200029
--------------------------------------------------
sentence_id: SENT_22548
--------------------------------------------------
entity_id: UENT_19918
--------------------------------------------------
char_start: 5
--------------------------------------------------
char_end: 15
--------------------------------------------------
token_start: 2
--------------------------------------------------
token_end: 3
--------------------------------------------------
entity_local_id: ENT_200030
--------------------------------------------------
sentence_id: SENT_22548
--------------------------------------------------
entity_id: UENT_22307
--------------------------------------------------
char_start: 19
--------------------------------------------------
char_end: 23
--------------------------------------------------
token_start: 5
--------------------------------------------------
token_end: 5
-------------------------------------------

: 